In [0]:
dbutils.widgets.text("wave", "", "Wave id")
dbutils.widgets.text("continuation", "1", "Continuation pass (1-based)")
dbutils.widgets.text("run_id", "", "Job run id")
dbutils.widgets.text("build_root", "", "Build root (project Git folder)")
dbutils.widgets.text("manifest_path", "", "Manifest path (waves.json)")

wave = dbutils.widgets.get("wave").strip()
continuation = dbutils.widgets.get("continuation").strip() or "1"
run_id = dbutils.widgets.get("run_id").strip()
build_root = dbutils.widgets.get("build_root").strip()
manifest_path = dbutils.widgets.get("manifest_path").strip()

if not wave or not run_id or not build_root or not manifest_path:
    raise ValueError(f"wave, run_id, build_root and manifest_path are required "
                     f"(got wave={wave!r}, run_id={run_id!r}, build_root={build_root!r}, manifest_path={manifest_path!r})")

print(f"wave={wave} continuation={continuation} run_id={run_id}")

In [0]:
import json
import os
import sys

# waves_manifest.py and session_status.py sit next to this notebook (cwd for workspace notebooks).
sys.path.insert(0, os.getcwd())
from waves_manifest import is_archived
from session_status import incomplete_sessions

with open(manifest_path) as f:
    manifest = json.load(f)

wave_cfg = next((w for w in manifest.get("waves") or [] if str(w.get("wave_id")) == wave), None)
if wave_cfg is None:
    raise ValueError(f"Wave {wave!r} not found in {manifest_path}")
if is_archived(wave_cfg):
    raise ValueError(f"Wave {wave!r} is archived; re-run setup so the build job has no tasks for it")

status_dir = os.path.join(build_root, manifest["status_dir"], run_id)
incomplete = incomplete_sessions(wave_cfg, status_dir)

print(f"status dir: {status_dir}")
print(f"sessions in wave: {len(wave_cfg.get('sessions') or [])}, incomplete: {len(incomplete)}")
for i in incomplete:
    print(f"  {i['session_id']}: {i['reason']} ({i['title']})")

# for_each inputs for w<wave>_continue[k]: same shape as the w<wave>_sessions inputs.
inputs = [{"session_id": i["session_id"], "title": i["title"]} for i in incomplete]
try:
    dbutils.jobs.taskValues.set(key="incomplete", value=inputs)
    dbutils.jobs.taskValues.set(key="has_incomplete", value="true" if inputs else "false")
except Exception:
    pass  # interactive run: no job context, taskValues unavailable

dbutils.notebook.exit(json.dumps({"wave": wave, "continuation": continuation, "incomplete": incomplete}))